# Solutions — Bakehouse Franchises & Sales (Medium 09)

**Datasets:** `sales_franchises`, `sales_transactions`

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

franchises   = spark.table("samples.bakehouse.sales_franchises")
transactions = spark.table("samples.bakehouse.sales_transactions")

## Solution 1 — Revenue & Transaction Count per Franchise

In [ ]:
result_1 = (
    transactions
    .join(franchises, on="franchiseID")
    .groupBy("franchiseID","name","country")
    .agg(
        F.round(F.sum("totalPrice"), 2).alias("total_revenue"),
        F.count("*").alias("transaction_count")
    )
    .orderBy(F.col("total_revenue").desc())
)

## Solution 2 — Top 5 Franchises by Total Revenue

In [ ]:
result_2 = (
    transactions
    .join(franchises, on="franchiseID")
    .groupBy("franchiseID","name","country","size")
    .agg(F.round(F.sum("totalPrice"), 2).alias("total_revenue"))
    .orderBy(F.col("total_revenue").desc())
    .limit(5)
)

## Solution 3 — Revenue per Franchise Size

In [ ]:
result_3 = (
    transactions
    .join(franchises, on="franchiseID")
    .groupBy("size")
    .agg(
        F.round(F.sum("totalPrice"), 2).alias("total_revenue"),
        F.countDistinct("franchiseID").alias("franchise_count")
    )
    .orderBy(F.col("total_revenue").desc())
)

## Solution 4 — Top 3 Franchises by Revenue within Country

In [ ]:
w = Window.partitionBy("country").orderBy(F.col("total_revenue").desc())
result_4 = (
    transactions
    .join(franchises, on="franchiseID")
    .groupBy("country","name")
    .agg(F.round(F.sum("totalPrice"), 2).alias("total_revenue"))
    .withColumn("rank", F.rank().over(w))
    .filter(F.col("rank") <= 3)
    .orderBy("country","rank")
)

## Solution 5 — Best-Selling Product per Franchise

In [ ]:
# Why: row_number per franchise ordered by quantity desc gives rank 1 = best seller
w = Window.partitionBy("franchiseID").orderBy(F.col("total_quantity").desc())
result_5 = (
    transactions
    .join(franchises, on="franchiseID")
    .groupBy("franchiseID","name","product")
    .agg(F.sum("quantity").alias("total_quantity"))
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select(
        "franchiseID","name",
        F.col("product").alias("best_product"),
        "total_quantity"
    )
)

## Solution 6 — Avg Transaction Value per Franchise vs Overall Avg

In [ ]:
overall_avg = transactions.agg(F.avg("totalPrice")).collect()[0][0]
result_6 = (
    transactions
    .join(franchises, on="franchiseID")
    .groupBy("franchiseID","name")
    .agg(F.round(F.avg("totalPrice"), 2).alias("avg_transaction_value"))
    .withColumn(
        "above_overall_avg",
        F.col("avg_transaction_value") > round(overall_avg, 2)
    )
    .orderBy(F.col("avg_transaction_value").desc())
)

## Solution 7 — Franchises that Sold Every Unique Product

In [ ]:
total_products = transactions.select("product").distinct().count()
result_7 = (
    transactions
    .join(franchises, on="franchiseID")
    .groupBy("franchiseID","name")
    .agg(F.countDistinct("product").alias("products_sold"))
    .withColumn("total_products", F.lit(total_products))
    .filter(F.col("products_sold") == F.col("total_products"))
    .orderBy("franchiseID")
)